# SIM cost anomaly model, connected to the live system

This notebook runs **the same code** as the nightly `ML pipeline` GitHub Actions workflow (`src/ml-pipeline`).
It reads billing history from the deployed API, evaluates a candidate model against the transparent rule on the most
recent months (which the model never sees in training), and can register the candidate so it appears on the website's
**Asset intelligence** page.

* Colab is the lab; the scheduled workflow is production. Both talk to the API, never to the database.
* A registered model is only a **candidate**. An administrator activates it in the web app after comparing it with the rule.
* Two modes: `offline` uses the synthetic generator (no credentials needed); `live` uses the API and the integration token.

In [ ]:
import subprocess, sys
from pathlib import Path
REPO = Path('/content/sim-inventory-management-system')
BRANCH = 'main'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/RURIGIMK/sim-inventory-management-system.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'src/ml-pipeline/requirements.txt')], check=True)
sys.path.insert(0, str(REPO / 'src/ml-pipeline'))
sys.path.insert(0, str(REPO / 'tests/ml-pipeline'))
print('Ready')

## Configure
Leave `MODE = 'offline'` to explore with synthetic data. For `live`, set the API URL; the token is asked for and never stored in the notebook.

In [ ]:
MODE = 'offline'          # 'offline' or 'live'
API_BASE_URL = ''          # e.g. the Render API URL, https://...
REGISTER_CANDIDATE = False # live only: register the candidate and publish its scores
TEST_MONTHS = 5
FLAG_RATE = 0.05

In [ ]:
import json, os, getpass
from sim_ml.features import build_frame
from sim_ml import model as m
from sim_ml.client import ApiClient

if MODE == 'live':
    token = getpass.getpass('Integration token: ')
    os.environ['SIM_INTEGRATION_TOKEN'] = token
    client = ApiClient(API_BASE_URL, token)
    client.wait_until_healthy()
    dataset = client.dataset()
else:
    from conftest import make_dataset
    client, dataset = None, make_dataset()

frame = build_frame(dataset)
print(f"{len(frame)} SIM-months, {frame['simId'].nunique()} SIMs, {frame['period'].nunique()} months")
frame[['simId', 'periodText', 'amount', 'contracted', 'dataUsageMb', 'voiceMinutes'] + m.MODEL_FEATURES].tail(10)

## Evaluate on months the model has not seen
The model is trained on earlier months and tested on the most recent ones. It is compared with the rule, and with
simply ranking bills by distance from the contract while flagging the same number of rows. Labels come from human
reviews once there are enough, otherwise from the synthetic scenarios.

In [ ]:
from sim_ml.run import choose_labels
import pandas as pd
label = choose_labels(frame, min_reviews=30)
print('Labels used:', label)
result = m.evaluate(frame, label, TEST_MONTHS, FLAG_RATE) if label else {'evaluated': False}
if result.get('evaluated'):
    rows = {k: result[k] for k in ['candidate', 'rule', 'ratio_ranked_same_budget'] if k in result}
    display(pd.DataFrame(rows).T[['flagged', 'precision', 'recall', 'f1', 'false_positive_rate']])
    if 'recall_by_scenario' in result:
        display(pd.DataFrame(result['recall_by_scenario']).T)
else:
    print(result)

In [ ]:
import matplotlib.pyplot as plt
bundle = m.fit(frame, FLAG_RATE)
frame['score'] = bundle.score(frame)
fig, ax = plt.subplots(figsize=(8, 3))
normal = frame[frame['label_synthetic'] != 1]['score'] if 'label_synthetic' in frame else frame['score']
ax.hist(normal, bins=40, alpha=.6, label='other months')
if (frame.get('label_synthetic') == 1).any():
    ax.hist(frame[frame['label_synthetic'] == 1]['score'], bins=20, alpha=.8, label='known anomalies')
ax.axvline(bundle.threshold, color='k', linestyle='--', label='flag threshold')
ax.set_xlabel('anomaly score (higher = more unusual)'); ax.legend(); plt.show()
flagged = frame[frame['score'] >= bundle.threshold].sort_values('score', ascending=False).head(10)
for _, row in flagged.iterrows():
    print(row['simId'], row['periodText'], '|', '; '.join(m.explain(bundle, row)))

## Register the candidate (live only)
This runs the full pipeline exactly as the nightly workflow does: it registers a candidate, publishes its scores and
refreshes the active model's scores. Nothing becomes active until an administrator activates it on the website.

In [ ]:
from sim_ml.run import main
if MODE == 'live' and REGISTER_CANDIDATE:
    main(['--api-base', API_BASE_URL, '--summary-json', 'run-summary.json'])
    print(open('run-summary.json').read())
else:
    print('Skipped: set MODE = "live" and REGISTER_CANDIDATE = True to register a candidate.')